#LLM

import library files

In [11]:
!pip install langchain_community -q
!pip install langchain-experimental -q
!pip install pdfplumber -q
!pip install faiss-cpu -q
!pip install langchain-openai -q

Step 1: Import Required Modules

In [12]:
import os
from langchain_openai import ChatOpenAI # Updated import
from langchain_core.messages import HumanMessage

Step 2: Set OpenRouter API Key and Endpoint

In [13]:
# Paste your OpenRouter API Key below
os.environ["OPENAI_API_KEY"] = "YOUR_OPENROUTER_API_KEY"
# OpenRouter Base URL
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

Step 3: Initialize the LLM

In [14]:
llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0.7,
    openai_api_base=os.environ["OPENAI_API_BASE"],
    openai_api_key=os.environ["OPENAI_API_KEY"],
    request_timeout=60,
    verbose=True
)

Step 4: Start Chat Loop

In [17]:
while True:
    # Take user input
    query = input("\nYou: ")
    # Exit condition
    if query.lower() in ["exit", "quit", "bye"]:
        print("\nGoodbye!")
        break
    # Send query to the model
    response = llm.invoke([
        HumanMessage(content=query)
    ])
    # Print AI response
    print("\nAssistant:", response.content)


You: hello

Assistant: Hello! How canI help you today?

You: can you tell me information about vijay

Assistant: Sure! “Vijay” is a fairly common name, and there are several well‑known people with that name. Could you let me know which Vijay you’re interested in? Here are a few of the most prominent ones:

| Name | Field / Notable For | Quick Summary |
|------|---------------------|---------------|
| **Joseph Vijay** (often just “Vijay”) | Tamil cinema (Indian film) | One of the biggest stars in Kollywood. Known for movies like *Theri*, *Sarkar*, *Bigil*, and *Master*. He’s also a philanthropist and has a large fan base in South India. |
| **Vijay Singh** | Professional golf | An Indian‑born golfer who has played on the Asian Tour and the European Tour. He’s known for his consistent tournament performances. |
| **Vijay Hazare** | Cricket (historical) | An Indian cricketer from the 1960s–70s who later became a noted administrator and commentator. The Vijay Hazare Trophy (one‑day domest

# RAG

import libraries

STEP 1: Install Required Libraries

In [18]:
!pip install langchain-classic

STEP 2: Import Required Modules

In [19]:
import os
from langchain_classic.chat_models import ChatOpenAI # OpenAI/OpenRouter Chat Model
from langchain_classic.chains import RetrievalQA # Retrieval QA Chain
from langchain_classic.chains.llm import LLMChain # LLM Chain
from langchain_classic.chains.combine_documents.stuff import StuffDocumentsChain
# Document Combining Chain
from langchain_classic.prompts import PromptTemplate # Prompt Template
from langchain_community.document_loaders import PDFPlumberLoader # PDF Loader
from langchain_experimental.text_splitter import SemanticChunker # Semantic Text Splitter
from langchain_community.embeddings import HuggingFaceEmbeddings # HuggingFace Embedding Model
from langchain_community.vectorstores import FAISS # FAISS Vector Database

STEP 3: Set OpenRouter API Key and Base URL

In [20]:
# Paste your OpenRouter API Key here
os.environ["OPENAI_API_KEY"] = "YOUR_OPENROUTER_API_KEY"
# OpenRouter API Endpoint
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

STEP 4: Load Resume PDF

In [22]:
# Load the PDF file
loader = PDFPlumberLoader(
    "/content/resume/ARVIND_R_K_FlowCV_Resume_2026-05-07.pdf"
)
# Extract text from PDF
docs = loader.load()
# Print total pages loaded
print("Pages loaded:", len(docs))

Pages loaded: 1


STEP 5: Split PDF Content into Chunks

In [23]:
# Initialize embedding model for semantic chunking
text_splitter = SemanticChunker(
    HuggingFaceEmbeddings()
)
# Split documents into meaningful chunks
documents = text_splitter.split_documents(docs)
# Print total chunks created
print("Chunks created:", len(documents))

/tmp/ipykernel_1231/2406718417.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  HuggingFaceEmbeddings()
/tmp/ipykernel_1231/2406718417.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks created: 2


STEP 6: Create Embeddings and Vector Database

In [24]:
# Load embedding model
embedder = HuggingFaceEmbeddings()
# Create FAISS vector database
vector = FAISS.from_documents(
    documents,
    embedder
)
# Create retriever
retriever = vector.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

/tmp/ipykernel_1231/4257189516.py:2: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedder = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


STEP 7: Initialize OpenRouter LLM

In [25]:
llm = ChatOpenAI(
    # Free OpenRouter Model
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    # Creativity level
    temperature=0.7,
    # OpenRouter Base URL
    openai_api_base="https://openrouter.ai/api/v1",
    # API Key
    openai_api_key=os.environ["OPENAI_API_KEY"],
    # Timeout in seconds
    request_timeout=60,
)


/tmp/ipykernel_1231/3419628468.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


STEP 8: Create Prompt Template

In [26]:
prompt = """
You are a helpful assistant.

Use the following pieces of context to answer the question at the end.

Answer only using the context and be concise (3–4 sentences).

Context:
{context}

Question:
{question}

Answer:
"""
# Convert prompt into LangChain template
QA_CHAIN_PROMPT = PromptTemplate.from_template(prompt)


STEP 9: Create LLM Chain

In [27]:
llm_chain = LLMChain(
    # LLM Model
    llm=llm,
    # Prompt Template
    prompt=QA_CHAIN_PROMPT,
    # Show logs
    verbose=True
)

/tmp/ipykernel_1231/2204679672.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(


STEP 10: Create Document Prompt

In [28]:
document_prompt = PromptTemplate(
    input_variables=[
        "page_content",
        "source"
    ],
    template="""
    Context:
    content:{page_content}
    source:{source}
    """
)

STEP 11: Combine Retrieved Documents

In [29]:
combine_documents_chain = StuffDocumentsChain(
    # Main LLM Chain
    llm_chain=llm_chain,
    # Variable name used in prompt
    document_variable_name="context",
    # Document formatting prompt
    document_prompt=document_prompt,
    # Optional callbacks
    callbacks=None,
)

/tmp/ipykernel_1231/2355887479.py:1: LangChainDeprecationWarning: The class `StuffDocumentsChain` was deprecated in LangChain 0.2.13 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build new RAG flows with `create_agent` and a retrieval tool. See https://docs.langchain.com/oss/python/langchain/rag
  combine_documents_chain = StuffDocumentsChain(


STEP 12: Create Retrieval QA Pipeline

In [30]:
qa = RetrievalQA(
    # Document Combiner
    combine_documents_chain=combine_documents_chain,
    # Vector Retriever
    retriever=retriever,
    # Return source documents
    return_source_documents=True,
    # Show execution logs
    verbose=True,
)

/tmp/ipykernel_1231/10775398.py:1: LangChainDeprecationWarning: The class `RetrievalQA` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build new RAG flows with `create_agent` and a retrieval tool. See https://docs.langchain.com/oss/python/langchain/rag
  qa = RetrievalQA(


STEP 13: Ask Questions

In [32]:
# User question
query = "What are the candidate skills and his name"
# Run QA pipeline
result = qa(query)
# Print final answer
print("\nAnswer:")
print(result["result"])



> Entering new RetrievalQA chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are a helpful assistant.

Use the following pieces of context to answer the question at the end.

Answer only using the context and be concise (3–4 sentences).

Context:

    Context:
    content:ARVIND R K
arvindguru83 @gmail.com 8778901907 Chennai, India ArvindRK github.com/ArvindGuruRK
SUMMARY
A motivated fresher seeking an IT role, with a solid foundation in various tools and technologies. I have gained hands-on
experience through internships and personal projects, showcasing my ability to learn quickly and adapt. Eager to apply my
existing skills in an IT company, I am committed to contributing to organizational growth and enhancing my knowledge
while gaining valuable industry experience. EDUCATION
Rajalakshmi Institute of Technology , Bachelor of Engineering in Computer Science 2021 – 2025
8.14 CGPA Chennai, India
Velammal Vidhyashram CBSE , X and XII 2008 – 2021
77.8 Percentage